# Introduction #

In this document the procedure used to get the dataset will be explained in detail

## FIRMS data recollection ##

First, we started by downloading historical thermal anomaly records. FIRMS offers them indexed by sensor, then year and finally country. Here is the website for that: https://firms.modaps.eosdis.nasa.gov/country/

So as to show their format, below is the example of the MODIS sensor 2020 anomalies for Uruguay:

In [ ]:
import pandas as pd

# Load the MODIS wildfire data
csv_path = "data/modis_2020_Uruguay.csv"
df = pd.read_csv(csv_path)
df.head(10)

From this data, the important ones are:
- latitude
- longitude
- acq_date
- acq_time

Multiple of these files are downloaded, and then merged into a new csv that contains all the anomalies, over which it is possible to iterate.
Then, a GEE query is performed over these elements to perform a query. A Sentinel-2 image was looked for, near the time and place of the anomaly, to check for a possible fire. Almost 100,000 images were downladed and manually reviewed, searching for fires.
Sometimes FIRMS creates more than one alert for the same fire, in some cases the exact same image is duplicated, and in some others there is a little variation in time/location.
After registering the data of the images that corresponded to fires and a process of deduplication, the following dataset was created. Some auxiliary fields like Country were added

In [ ]:
import pandas as pd

# Load the MODIS wildfire data
csv_path = "data/firms_features_merged_last_matched.csv"
df = pd.read_csv(csv_path)
df.head(10)

## Recreating Dataset ##
Below is a code that can be used to download the images in the dataset

In [ ]:
import os
import pandas as pd
from pathlib import Path
import ee
from google.cloud import storage


def load_env_file(path: str) -> dict:
    env_values = {}
    if not os.path.exists(path):
        return env_values
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            env_values[key.strip()] = value.strip().strip('"').strip("'")
    return env_values


ENV_PATH = Path("config/.env")
env_values = load_env_file(ENV_PATH)
os.environ.update(env_values)

BANDS_TO_EXTRACT = ["B4", "B3", "B2", "B8", "B11", "B12"]
CSV_PATH = "data/firms_features_merged_last_matched.csv"
OUTPUT_DIR = "fire_downloads"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "downloaded_images.csv")
BUCKET_NAME = os.environ.get("BUCKET_NAME", "fire_dataset_2")
GEE_PROJECT = os.environ.get("GEE_PROJECT", "fire-detection-uruguay")

"""os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Initialize()

storage_client = storage.Client()
_ = storage_client.bucket(BUCKET_NAME)

records = []

for idx, row in df.iterrows():
    lat = row["latitude"]
    lon = row["longitude"]
    image_date = row["image_date"]
    thumbnail_file = row["thumbnail_file"]
    country = str(row["country"]).strip()

    target_dt = pd.to_datetime(image_date)
    start_dt = target_dt - pd.Timedelta(hours=10)
    end_dt = target_dt + pd.Timedelta(hours=10)

    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(2000)

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate(ee.Date(start_dt.to_pydatetime()), ee.Date(end_dt.to_pydatetime()))
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
        .select(BANDS_TO_EXTRACT)
        .sort("system:time_start")
    )

    count = collection.size().getInfo()
    if count == 0:
        print(f"No suitable image found for {thumbnail_file}")
        continue

    image_list = collection.toList(count)
    target_ms = int(target_dt.timestamp() * 1000)
    best_image = None
    best_delta = None

    for i in range(count):
        img = ee.Image(image_list.get(i))
        img_time = img.get("system:time_start").getInfo()
        delta = abs(img_time - target_ms)
        if best_delta is None or delta < best_delta:
            best_delta = delta
            best_image = img

    if best_image is None:
        continue

    try:
        image_id = best_image.get("system:index").getInfo()
    except Exception:
        image_id = best_image.id().getInfo()

    safe_country = country.replace(" ", "_")
    name = f"{safe_country}_{idx}"

    export_task = ee.batch.Export.image.toCloudStorage(
        image=best_image,
        bucket=BUCKET_NAME,
        fileNamePrefix=f"{OUTPUT_DIR}/{name}",
        region=region,
        scale=10,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True},
        maxPixels=1e13,
    )
    export_task.start()

    records.append(
        {
            "latitude": lat,
            "longitude": lon,
            "acquisition_time": target_dt.isoformat(),
            "country": country,
            "image_id": image_id,
            "name": name,
        }
    )
    print(f"Queued {name} from {image_id}")

pd.DataFrame(records).to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(records)} records to {OUTPUT_CSV}")
"""

In [ ]:
import os
from pathlib import Path
import pandas as pd
import ee
import requests


def load_env_file(path: str) -> dict:
    env_values = {}
    if not os.path.exists(path):
        return env_values
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            env_values[key.strip()] = value.strip().strip('"').strip("'")
    return env_values


def gee_initialization(project: str | None = None) -> None:
    env_values = load_env_file(Path("config/.env"))
    os.environ.update(env_values)

    gee_project = project or os.environ.get("GEE_PROJECT")
    try:
        ee.Initialize(project=gee_project)
    except Exception:
        ee.Initialize()


def read_alert_row(df: pd.DataFrame, index: int) -> dict | None:
    if index < 0 or index >= len(df):
        return None

    row = df.iloc[index]
    target_dt = pd.to_datetime(row["image_date"])

    return {
        "lat": float(row["latitude"]),
        "lon": float(row["longitude"]),
        "datetime": target_dt,
        "country": str(row["country"]).strip(),
        "thumbnail_file": row["thumbnail_file"],
    }


def gee_get_image(lat: float, lon: float, target_dt: pd.Timestamp):
    print(f"BANDAS A SACAR SON {BANDS_TO_EXTRACT}")

    start_dt = target_dt - pd.Timedelta(hours=10)
    end_dt = target_dt + pd.Timedelta(hours=10)

    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(2000)

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate(ee.Date(start_dt.to_pydatetime()), ee.Date(end_dt.to_pydatetime()))
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
        .select(BANDS_TO_EXTRACT)
        .sort("system:time_start")
    )

    count = collection.size().getInfo()
    if count == 0:
        return None, None

    image_list = collection.toList(count)
    target_ms = int(target_dt.timestamp() * 1000)
    best_image = None
    best_delta = None

    for i in range(count):
        img = ee.Image(image_list.get(i))
        img_time = img.get("system:time_start").getInfo()
        delta = abs(img_time - target_ms)
        if best_delta is None or delta < best_delta:
            best_delta = delta
            best_image = img

    return best_image, region


def gee_download_image(best_image, name: str, output_dir: str, region, scale: int = 10) -> None:
    task = ee.batch.Export.image.toDrive(
        image=best_image,
        description=name,
        folder=output_dir,
        region=region,
        scale=scale,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True},
        maxPixels=1e13,
    )
    task.start()


def gee_download_local(point, image, output_filename: str, scale: int = 10) -> bool:
    region = point.buffer(2000).bounds()

    download_params = {
        "region": region,
        "scale": scale,
        "format": "GEO_TIFF",
    }

    try:
        url = image.clip(region).getDownloadURL(download_params)
        response = requests.get(url, timeout=60)

        if response.status_code != 200:
            print(f"[ERROR] HTTP {response.status_code}")
            return False

        if "image/tiff" not in response.headers.get("Content-Type", ""):
            print("[ERROR] Not a TIFF response")
            return False

        if len(response.content) < 1000:
            print("[ERROR] File too small — likely invalid")
            return False

        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with open(output_filename, "wb") as f:
            f.write(response.content)

        return True

    except Exception as e:
        print(f"[ERROR] {e}")
        return False


def gee_download_thumb_local(point, image, output_filename: str, width: int = 1024, height: int = 1024) -> bool:
    region = point.buffer(2000).bounds()

    thumb_params = {
        "region": region,
        "dimensions": f"{width}x{height}",
        "format": "png",
        "bands": ["B4", "B3", "B2"],
        "min": 0,
        "max": 3000,
        "gamma": 1.2,
    }
    # TODO VER ESTOS NÚMEROS MÁGICOS

    try:
        url = image.clip(region).getThumbURL(thumb_params)
        response = requests.get(url, timeout=60)

        if response.status_code != 200:
            print(f"[ERROR] HTTP {response.status_code}")
            return False

        if "image/png" not in response.headers.get("Content-Type", ""):
            print("[ERROR] Not a PNG response")
            return False

        if len(response.content) < 1000:
            print("[ERROR] File too small — likely invalid")
            return False

        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with open(output_filename, "wb") as f:
            f.write(response.content)

        return True

    except Exception as e:
        print(f"[ERROR] {e}")
        return False


def save_image_info(records: list[dict], lat: float, lon: float, target_dt: pd.Timestamp, country: str, image_id: str, name: str) -> list[dict]:
    records.append(
        {
            "latitude": lat,
            "longitude": lon,
            "image_date": target_dt.isoformat(),
            "country": country,
            "image_id": image_id,
            "name": name,
        }
    )
    return records

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

# Example workflow
gee_initialization()

DOWNLOAD_DIR = "fire_downloads"
PNG_DOWNLOAD_DIR = "fire_downloads_png"
GET_PNG = False
NUM_WORKERS = 4
OUTPUT_CSV = os.path.join(DOWNLOAD_DIR, "downloaded_images.csv")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

csv_path = "data/firms_features_merged_last_matched.csv"
df = pd.read_csv(csv_path)

csv_columns = [
    "latitude",
    "longitude",
    "acquisition_datetime",
    "country",
    "image_id",
    "name",
]

if not os.path.exists(OUTPUT_CSV):
    pd.DataFrame(columns=csv_columns).to_csv(OUTPUT_CSV, index=False)

csv_lock = Lock()


def process_alert_row(idx: int):
    alert = read_alert_row(df, idx)
    if alert is None:
        return None

    best_image, _ = gee_get_image(alert["lat"], alert["lon"], alert["datetime"])
    if best_image is None:
        print(f"No suitable image found for row {idx}")
        return None

    try:
        image_id = best_image.get("system:index").getInfo()
    except Exception:
        image_id = best_image.id().getInfo()

    safe_country = alert["country"].replace(" ", "_")
    name = f"{safe_country}_{idx}"
    output_filename = os.path.join(DOWNLOAD_DIR, f"{name}.tif")
    thumb_filename = os.path.join(PNG_DOWNLOAD_DIR, f"{name}.png")

    point = ee.Geometry.Point([alert["lon"], alert["lat"]])
    #success_tif = gee_download_local(point, best_image, output_filename) TODO DESCOMENTAR

    #if success_tif:
     #   print(f"Downloaded {output_filename}")
    #else:
     #   print(f"Failed to download {output_filename}") TODO DESCOMENTAR PÁRRAFO

    if GET_PNG:
        os.makedirs(PNG_DOWNLOAD_DIR, exist_ok=True)
        success_png = gee_download_thumb_local(point, best_image, thumb_filename)
        if success_png:
            print(f"Downloaded {thumb_filename}")
        else:
            print(f"Failed to download {thumb_filename}")
    else:
        print("Skipping PNG download")

    print(f"Queued {name} from {image_id}")
    record = {
        "latitude": alert["lat"],
        "longitude": alert["lon"],
        "acquisition_datetime": alert["datetime"].isoformat(),
        "country": alert["country"],
        "image_id": image_id,
        "name": name,
    }

    with csv_lock:
        pd.DataFrame([record]).to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
    return record


with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    list(executor.map(process_alert_row, range(len(df))))

print(f"Finished writing records to {OUTPUT_CSV}")

In [ ]:
#Auxiliary functions for downloading no fire images

import random
import pandas as pd

random.seed(42)


def get_offset_target_dt(target_dt: pd.Timestamp, tried_offsets: set, max_n: int = 20):
    available = [n for n in range(1, max_n + 1) if n not in tried_offsets]
    if not available:
        return None, None

    n = random.choice(available)
    tried_offsets.add(n)

    offset_days = 5 * n
    new_target_dt = target_dt - pd.Timedelta(days=offset_days)
    return new_target_dt, offset_days


def gee_get_image_with_backoff(lat: float, lon: float, target_dt: pd.Timestamp, max_retries: int = 10):
    tried_offsets = set()

    for attempt in range(1, max_retries + 1):
        new_target_dt, offset_days = get_offset_target_dt(target_dt, tried_offsets)

        if new_target_dt is None:
            print("Ran out of distinct offsets to try.")
            break

        print(f"[Attempt {attempt}] Trying target_dt shifted -{offset_days} days -> {new_target_dt}")

        best_image, region = gee_get_image(lat, lon, new_target_dt)

        if best_image is not None:
            print(f"Found image on attempt {attempt} with offset -{offset_days} days.")
            return best_image, region

    print("All retries exhausted, no image found.")
    return None, None

def read_row_no_fire(df: pd.DataFrame, index: int) -> dict | None:
    if index < 0 or index >= len(df):
        return None

    row = df.iloc[index]
    target_dt = pd.to_datetime(row["acquisition_datetime"])

    return {
        "lat": float(row["latitude"]),
        "lon": float(row["longitude"]),
        "acquisition_datetime": target_dt,
        "country": str(row["country"]).strip(),
    }

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

# Example workflow
gee_initialization()

DOWNLOAD_DIR = "no_fire_downloads"
PNG_DOWNLOAD_DIR = "fire_downloads_png"
GET_PNG = True
NUM_WORKERS = 4
OUTPUT_CSV = os.path.join("final_data", "downloaded_images.csv")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

csv_path = "final_data/downloaded_images_sorted.csv"
df = pd.read_csv(csv_path)

csv_columns = [
    "latitude",
    "longitude",
    "acquisition_datetime",
    "country",
    "image_id",
    "name",
]

if not os.path.exists(OUTPUT_CSV):
    pd.DataFrame(columns=csv_columns).to_csv(OUTPUT_CSV, index=False)

csv_lock = Lock()


def process_alert_row(idx: int):
    alert = read_row_no_fire(df, idx)
    if alert is None:
        return None

    while True: # fetch image until retrieved
        best_image, _ = gee_get_image_with_backoff(alert["lat"], alert["lon"], alert["acquisition_datetime"])
        if best_image is not None:
            print(f"No suitable image found for row {idx}")
            continue
        else:
            break

    try:
        image_id = best_image.get("system:index").getInfo()
    except Exception:
        image_id = best_image.id().getInfo()

    safe_country = alert["country"].replace(" ", "_")
    name = f"{safe_country}_{idx}"
    output_filename = os.path.join(DOWNLOAD_DIR, f"{name}.tif")
    thumb_filename = os.path.join(PNG_DOWNLOAD_DIR, f"{name}.png")

    point = ee.Geometry.Point([alert["lon"], alert["lat"]])
    success_tif = gee_download_local(point, best_image, output_filename) 

    if success_tif:
        print(f"Downloaded {output_filename}")
    else:
        print(f"Failed to download {output_filename}")

    if GET_PNG:
        os.makedirs(PNG_DOWNLOAD_DIR, exist_ok=True)
        success_png = gee_download_thumb_local(point, best_image, thumb_filename)
        if success_png:
            print(f"Downloaded {thumb_filename}")
        else:
            print(f"Failed to download {thumb_filename}")
    else:
        print("Skipping PNG download")

    print(f"Queued {name} from {image_id}")
    record = {
        "latitude": alert["lat"],
        "longitude": alert["lon"],
        "acquisition_datetime": alert["datetime"].isoformat(),
        "country": alert["country"],
        "image_id": image_id,
        "name": name,
    }

    with csv_lock:
        pd.DataFrame([record]).to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
    return record


with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    list(executor.map(process_alert_row, range(len(df))))

print(f"Finished writing records to {OUTPUT_CSV}")

Found image on attempt 6 with offset -60 days.
No suitable image found for row 0
[Attempt 1] Trying target_dt shifted -75 days -> 2024-10-16 14:37:23.935000
BANDAS A SACAR SON ['B4', 'B3', 'B2', 'B8', 'B11', 'B12']
Found image on attempt 3 with offset -95 days.
No suitable image found for row 1
[Attempt 1] Trying target_dt shifted -30 days -> 2024-11-28 14:46:47.082000
BANDAS A SACAR SON ['B4', 'B3', 'B2', 'B8', 'B11', 'B12']
Found image on attempt 2 with offset -95 days.
No suitable image found for row 3
[Attempt 1] Trying target_dt shifted -80 days -> 2024-10-09 14:46:47.082000
BANDAS A SACAR SON ['B4', 'B3', 'B2', 'B8', 'B11', 'B12']
[Attempt 2] Trying target_dt shifted -80 days -> 2024-10-11 14:37:23.935000
BANDAS A SACAR SON ['B4', 'B3', 'B2', 'B8', 'B11', 'B12']
[Attempt 1] Trying target_dt shifted -45 days -> 2024-11-15 14:37:23.935000
BANDAS A SACAR SON ['B4', 'B3', 'B2', 'B8', 'B11', 'B12']
[Attempt 1] Trying target_dt shifted -40 days -> 2024-11-18 14:46:47.082000
BANDAS A SA